# **Data Cleaning in Python**

>Our objective is to clean the tmdb_movies dataset using python.

## Overview

>The dataset is a TMDB/IMDB movie dataset containing 10,869 rows and 21 columns, covering movie metadata such as budgets, revenues, cast, genres, release dates, and ratings. The cleaning process addressed structural issues (wrong data types, inconsistent date formats), missing value strategy (distinguishing true nulls from sentinel zeros), and data quality problems (duplicates, outliers, low-value columns) to produce an analysis-ready dataframe.

In [1]:
# Import libraries
import pandas as pd
import numpy as np 

# Ignore notebook warnings
import warnings
warnings.filterwarnings("ignore")

## Load the dataset

In [2]:
# load the dataset 

df = pd.read_csv("Hey-pythonista.csv")

print(f"original shape: {df.shape}")
print(df.dtypes)  

original shape: (10869, 21)
id                      float64
imdb_id                  object
popularity              float64
budget                  float64
revenue                 float64
original_title           object
cast                     object
homepage                 object
director                 object
tagline                  object
keywords                 object
overview                 object
runtime                 float64
genres                   object
production_companies     object
release_date             object
vote_count              float64
vote_average            float64
release_year            float64
budget_adj              float64
revenue_adj             float64
dtype: object


In [3]:
# Preview first 5 rows

df.head(4)

,id,imdb_id,popularity,budget,revenue,original_title,cast,homepage,director,tagline,...,overview,runtime,genres,production_companies,release_date,vote_count,vote_average,release_year,budget_adj,revenue_adj
0,135397.0,tt0369610,32.985763,150000000.0,1.513529e+09,Jurassic World,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,http://www.jurassicworld.com/,Colin Trevorrow,The park is open.,...,Twenty-two years after the events of Jurassic ...,124.0,Action|Adventure|Science Fiction|Thriller,Universal Studios|Amblin Entertainment|Legenda...,9/6/2015,5562.0,6.5,2015.0,137999939.3,1.392446e+09
1,76341.0,tt1392190,28.419936,150000000.0,3.784364e+08,Mad Max: Fury Road,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,http://www.madmaxmovie.com/,George Miller,What a Lovely Day.,...,An apocalyptic story set in the furthest reach...,120.0,Action|Adventure|Science Fiction|Thriller,Village Roadshow Pictures|Kennedy Miller Produ...,5/13/15,6185.0,7.1,2015.0,137999939.3,3.481613e+08
2,262500.0,tt2908446,13.112507,110000000.0,2.952382e+08,Insurgent,Shailene Woodley|Theo James|Kate Winslet|Ansel...,http://www.thedivergentseries.movie/#insurgent,Robert Schwentke,One Choice Can Destroy You,...,Beatrice Prior must confront her inner demons ...,119.0,Adventure|Science Fiction|Thriller,Summit Entertainment|Mandeville Films|Red Wago...,3/18/15,2480.0,6.3,2015.0,101199955.5,2.716190e+08
3,140607.0,tt2488496,11.173104,200000000.0,2.068178e+09,Star Wars: The Force Awakens,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,http://www.starwars.com/films/star-wars-episod...,J.J. Abrams,Every generation has a story.,...,Thirty years after defeating the Galactic Empi...,136.0,Action|Adventure|Science Fiction|Fantasy,Lucasfilm|Truenorth Productions|Bad Robot,12/15/15,5292.0,7.5,2015.0,183999919.0,1.902723e+09


In [4]:
# Check data info

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10869 entries, 0 to 10868
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    10866 non-null  float64
 1   imdb_id               10856 non-null  object 
 2   popularity            10867 non-null  float64
 3   budget                10866 non-null  float64
 4   revenue               10866 non-null  float64
 5   original_title        10866 non-null  object 
 6   cast                  10790 non-null  object 
 7   homepage              2936 non-null   object 
 8   director              10822 non-null  object 
 9   tagline               8042 non-null   object 
 10  keywords              9373 non-null   object 
 11  overview              10862 non-null  object 
 12  runtime               10866 non-null  float64
 13  genres                10843 non-null  object 
 14  production_companies  9836 non-null   object 
 15  release_date       

>Loooking at the data rows we  an observe some inconsitent data type and some null values

## Data Cleaning

>We shall carry out the following data cleaning steps

## 1. Drop Duplicate Rows

>Duplicate records inflate row counts and skew aggregations (e.g. mean revenue, genre frequency counts). Even 2 duplicates are removed as a matter of data integrity best practice.

In [5]:
# Check for duplicate
duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicates}")

# drop duplicate rows
df.drop_duplicates(inplace=True)
print(f"shape after dropping duplicates: {df.shape}")


Number of duplicate rows: 2
shape after dropping duplicates: (10867, 21)


In [6]:
# verify duplicate rows dropped ────────────────────────────────────
print(f"duplicate rows remaining : {df.duplicated().sum()}")   # should be 0
print(f"current row count        : {len(df)}")                 # should be 10867
 

duplicate rows remaining : 0
current row count        : 10867


## 2. Drop Rows Where All Critical Fields Are Null

>Dropped rows where every critical column — `id, imdb_id, original_title, release_date, runtime, vote_count, vote_average` — was simultaneously null.

>A row missing all identifying and analytical fields carries zero informational value and cannot be recovered or imputed. Dropping only when all critical fields are null (not just one) is a conservative approach that avoids over-dropping rows that are partially populated.

In [7]:
# drop rows where all critical fields are null ──────────────────────────

# these are the columns that make a row meaningless without them
critical_cols = ["id", "imdb_id", "original_title", "release_date",
                 "runtime", "vote_count", "vote_average"]

df.dropna(subset=critical_cols, how="all", inplace=True)

print(f"shape after dropping all-null critical rows: {df.shape}")

shape after dropping all-null critical rows: (10865, 21)


In [8]:
# ── verify that all-null critical rows dropped ─────────────────────────────
 
critical_cols = ["id", "imdb_id", "original_title", "release_date",
                 "runtime", "vote_count", "vote_average"]
all_null_mask = df[critical_cols].isnull().all(axis=1)
print(f"rows where all critical cols are null : {all_null_mask.sum()}") 

rows where all critical cols are null : 0


## 3. Fix the id Column Type

>Converted id from float64 (e.g. 135397.0) to nullable Int64.

>IDs are identifiers, not measurements — storing them as floats is semantically wrong and risks silent precision loss on very large integers. Nullable Int64 preserves any remaining NaN values while using the correct integer type.


In [9]:
# id came in as float64 (e.g. 135397.0) — we will convert to nullable integer
df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")


In [10]:
# ── verify id column type ────────────────────────────
 
print(f"id dtype : {df['id'].dtype}")         
print(f"sample   : {df['id'].head().tolist()}")

id dtype : Int64
sample   : [135397, 76341, 262500, 140607, 168259]


## 4. Standardise release_date

>Parsed the `release_date` column with `pd.to_datetime(..., errors="coerce")` to unify two coexisting formats — Python `datetime` objects and date strings like `"5/13/15"` or `"12/15/15"`.

>Mixed types in a single column make time-based analysis (filtering by year, sorting chronologically, computing movie age) unreliable. Coercing to a single `datetime64` type makes all date operations consistent. Unparseable values become NaT rather than raising errors.

In [11]:
# the column is mixed: some cells are datetime objects, some are strings
# like "5/13/15" or "12/15/15" — we will coerce everything to a proper datetime
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

# rebuild release_year from the cleaned release_date so both columns agree
df["release_year"] = df["release_date"].dt.year.astype("Int64")

print("release_date nulls after fix:", df["release_date"].isnull().sum())

release_date nulls after fix: 6325


In [12]:
# ── verify release_date standardised ─────────────────────────────────
 
print(f"dtype          : {df['release_date'].dtype}")    # should be datetime64
print(f"null count     : {df['release_date'].isnull().sum()}")
print(f"sample values  :")
print(df[["release_date", "release_year"]].head(10).to_string())

dtype          : datetime64[ns]
null count     : 6325
sample values  :
  release_date  release_year
0   2015-09-06          2015
1          NaT          <NA>
2          NaT          <NA>
3          NaT          <NA>
4   2015-01-04          2015
5          NaT          <NA>
6          NaT          <NA>
7          NaT          <NA>
8          NaT          <NA>
9   2015-09-06          2015


In [13]:
# confirm no string dates remain — all non-null values should be Timestamps

non_null = df["release_date"].dropna()
non_timestamp = [v for v in non_null if not isinstance(v, pd.Timestamp)]
print(f"non-Timestamp values remaining : {len(non_timestamp)}")

non-Timestamp values remaining : 0


## 5. Replace Zero Budgets and Revenues with NaN

>Replaced `0` with `NaN` in `budget`, `revenue`, `budget_adj`, and `revenue_adj`.

>In this dataset, 0 is used as a sentinel value meaning "data not available" — not that a movie literally had no budget or earned nothing. Leaving zeros in place would severely distort descriptive statistics (mean, median) and any financial analysis or visualisation. Converting them to `NaN` makes the missingness explicit and excludes them from calculations automatically.

In [14]:
# zero is used as a sentinel for "unknown" — replace with NaN so statistics
# like mean / median are not dragged down to zero artificially
df["budget"]      = df["budget"].replace(0, np.nan)
df["revenue"]     = df["revenue"].replace(0, np.nan)
df["budget_adj"]  = df["budget_adj"].replace(0, np.nan)
df["revenue_adj"] = df["revenue_adj"].replace(0, np.nan)

print("budget  nulls after zero→NaN:", df["budget"].isnull().sum())
print("revenue nulls after zero→NaN:", df["revenue"].isnull().sum())

budget  nulls after zero→NaN: 5696
revenue nulls after zero→NaN: 6016


In [15]:
# ── verify zeros replaced with NaN in financial columns ───────────────
 
for col in ["budget", "revenue", "budget_adj", "revenue_adj"]:
    zero_count = (df[col] == 0).sum()
    null_count = df[col].isnull().sum()
    print(f"{col:<15} | zeros remaining: {zero_count}  | nulls: {null_count}")
# zeros remaining should be 0 for all four columns

budget          | zeros remaining: 0  | nulls: 5696
revenue         | zeros remaining: 0  | nulls: 6016
budget_adj      | zeros remaining: 0  | nulls: 5696
revenue_adj     | zeros remaining: 0  | nulls: 6016


## 6. Cast vote_count and runtime to Integer

>Converted `vote_count` and `runtime` from `float64` to nullable `Int64`.

>Half a vote or half a minute of runtime is not meaningful — these are inherently whole-number quantities. Using `Int64` (nullable) corrects the type while safely handling any remaining `NaN` values that `int` alone cannot represent.

In [16]:
# vote_count and runtime should be integers; we'll use Int64 to keep NaN support
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")
df["runtime"]    = pd.to_numeric(df["runtime"],    errors="coerce").astype("Int64")

# popularity and vote_average stay float — already correct type
df["popularity"]   = pd.to_numeric(df["popularity"],   errors="coerce")
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce")


In [17]:
# ── verify vote_count and runtime dtypes ──────────────────────────────

print(f"vote_count dtype : {df['vote_count'].dtype}")
print(f"runtime dtype    : {df['runtime'].dtype}") 
print(f"runtime sample   : {df['runtime'].head().tolist()}")
print(f"vote_count sample: {df['vote_count'].head().tolist()}")

vote_count dtype : Int64
runtime dtype    : Int64
runtime sample   : [124, 120, 119, 136, 137]
vote_count sample: [5562, 6185, 2480, 5292, 2947]


## 7. Clean Pipe-Separated Text Columns
>Stripped leading/trailing whitespace from each token inside the `cast`, `genres`, `keywords`, and `production_companies` columns, which use `|` as a delimiter.

>Inconsistent spacing (e.g. `"Action | Drama"` vs `"Action|Drama"`) causes the same value to be treated as different categories during string splitting, grouping, or one-hot encoding. Normalising the tokens ensures `"Action"` always equals `"Action"` regardless of surrounding spaces.

In [18]:
# cast, genres, keywords, production_companies all use "|" as delimiter
# strip whitespace around each token so "Action | Drama" → ["Action","Drama"]
pipe_cols = ["cast", "genres", "keywords", "production_companies"]

def clean_pipe_string(value):
    """strip whitespace from each token in a pipe-separated string."""
    if pd.isna(value):
        return np.nan
    return "|".join(token.strip() for token in str(value).split("|"))

for col in pipe_cols:
    df[col] = df[col].apply(clean_pipe_string)

In [19]:
# ── verify pipe-separated columns cleaned ────────────────────────────

pipe_cols = ["cast", "genres", "keywords", "production_companies"]
 
for col in pipe_cols:
    # check for tokens with leading or trailing spaces
    def has_space_around_pipe(val):
        if pd.isna(val):
            return False
        return any(token != token.strip() for token in str(val).split("|"))
 
    dirty = df[col].apply(has_space_around_pipe).sum()
    print(f"{col:<25} | tokens with surrounding spaces: {dirty}")

cast                      | tokens with surrounding spaces: 0
genres                    | tokens with surrounding spaces: 0
keywords                  | tokens with surrounding spaces: 0
production_companies      | tokens with surrounding spaces: 0


## 8. Strip Whitespace from String Columns
>Applied `.str.strip()` to `imdb_id`, `original_title`, `director`, `tagline`, `overview`, and `homepage`.

>Leading or trailing spaces cause silent mismatches in lookups, merges, and equality checks (e.g. `"Christopher Nolan"` ≠ `" Christopher Nolan"`). Stripping is a low-risk, high-value step on all free-text fields.

In [20]:
# strip whitespace 

str_cols = ["imdb_id", "original_title", "director",
            "tagline", "overview", "homepage"]

for col in str_cols:
    df[col] = df[col].str.strip()

## 9. Replace Empty Strings with NaN
>Replaced cells containing only whitespace or empty strings (`""`, `"   "`) across the entire dataframe with `NaN`.

>After stripping, some cells become empty strings rather than `NaN`. Empty strings are not the same as NaN in pandas — they are missed by `.isnull()`, skew value counts, and pollute analyses. Unifying them as `NaN` gives a single, consistent representation of missingness.

In [21]:
# replace empty strings with NaN across the whole frame

# some cells may hold "" or "   " after stripping — unify as NaN
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

In [22]:
# verify step 8 & 9, whitespace stripped, empty strings replaced ────────────
 
str_cols = ["imdb_id", "original_title", "director",
            "tagline", "overview"]
 
for col in str_cols:
    # check for leading/trailing spaces
    leading_trailing = df[col].dropna().apply(lambda x: x != x.strip()).sum()
    # check for empty strings
    empty = (df[col] == "").sum()
    print(f"{col:<20} | leading/trailing spaces: {leading_trailing}  | empty strings: {empty}")
 

imdb_id              | leading/trailing spaces: 0  | empty strings: 0
original_title       | leading/trailing spaces: 0  | empty strings: 0
director             | leading/trailing spaces: 0  | empty strings: 0
tagline              | leading/trailing spaces: 0  | empty strings: 0
overview             | leading/trailing spaces: 0  | empty strings: 0


## 10. Drop the homepage Column
>Removed the `homepage` column entirely.

>7,933 out of ~ 10,800 rows (~73%) had no homepage value — far too sparse to be analytically useful. Retaining a column with 73% missingness adds noise and memory overhead without contributing meaningful signal to any typical movie analysis task.

In [23]:
# 73 % of rows have no homepage (7,933 nulls out of ~10,800) — low signal
df.drop(columns=["homepage"], inplace=True)

print(f"columns after dropping homepage: {df.columns.tolist()}")

columns after dropping homepage: ['id', 'imdb_id', 'popularity', 'budget', 'revenue', 'original_title', 'cast', 'director', 'tagline', 'keywords', 'overview', 'runtime', 'genres', 'production_companies', 'release_date', 'vote_count', 'vote_average', 'release_year', 'budget_adj', 'revenue_adj']


In [24]:
# verify homepage column dropped

print(f"'homepage' in columns : {'homepage' in df.columns}")  # should be False
print(f"remaining columns     : {df.columns.tolist()}")

'homepage' in columns : False
remaining columns     : ['id', 'imdb_id', 'popularity', 'budget', 'revenue', 'original_title', 'cast', 'director', 'tagline', 'keywords', 'overview', 'runtime', 'genres', 'production_companies', 'release_date', 'vote_count', 'vote_average', 'release_year', 'budget_adj', 'revenue_adj']


## 11. Remove runtime Outliers
>Removed rows where `runtime` was less than 1 minute or greater than 600 minutes (10 hours).

>Values outside this range are almost certainly data entry errors, not real movies. A 0-minute runtime or a 700-minute runtime would distort distribution plots and summary statistics. The 1–600 minute window is generous enough to capture legitimate edge cases (short films and very long epics) while excluding nonsensical entries.

In [25]:
# movies shorter than 1 min or longer than 600 min are almost certainly errors

df = df[(df["runtime"].isna()) | ((df["runtime"] >= 1) & (df["runtime"] <= 600))]

print(f"shape after runtime outlier removal: {df.shape}")

shape after runtime outlier removal: (10831, 20)


In [26]:
# ── verify runtime outliers removed ──────────────────────────────────

valid = df["runtime"].dropna()
print(f"min runtime : {valid.min()} mins")   # should be >= 1
print(f"max runtime : {valid.max()} mins")   # should be <= 600
print(f"rows outside 1–600 min : {((valid < 1) | (valid > 600)).sum()}")  # should be 0

min runtime : 2 mins
max runtime : 566 mins
rows outside 1–600 min : 0


## 12. Remove Out-of-Range vote_average Values
>Removed rows where `vote_average` fell outside the valid 0–10 IMDB scale.
>
>IMDB ratings are bounded between 0 and 10. Any value outside this range is corrupt data and would distort rating-based analysis, filtering, and visualisations.

In [27]:
# imdb scale is 0–10; anything outside that is bad data

df = df[(df["vote_average"].isna()) |
        ((df["vote_average"] >= 0) & (df["vote_average"] <= 10))]

In [28]:
# Verify vote_average out-of-range removed 

valid_votes = df["vote_average"].dropna()
print(f"min vote_average : {valid_votes.min()}")   # should be >= 0
print(f"max vote_average : {valid_votes.max()}")   # should be <= 10
print(f"values outside 0–10 : {((valid_votes < 0) | (valid_votes > 10)).sum()}")  # should be 0

min vote_average : 1.5
max vote_average : 8.9
values outside 0–10 : 0


## 13. Reset the Index

>Reset the dataframe index with `reset_index(drop=True)` after all row removals.

>After dropping rows, the index contains gaps (e.g. 0, 1, 5, 9 …). A clean sequential index prevents off-by-one confusion during row-based operations and makes the dataframe easier to work with downstream.

In [29]:
# reset the index
df.reset_index(drop=True, inplace=True)

In [30]:
# ── verify index reset ──────────────────────────────────────────────

print(f"index start : {df.index[0]}")
print(f"index end   : {df.index[-1]}")
print(f"index gaps  : {df.index[-1] - (len(df) - 1)}")

index start : 0
index end   : 10830
index gaps  : 0


## 14. Final summary

In [31]:
# final summary fo clean data
 
print(f"rows    : {len(df)}")
print(f"columns : {len(df.columns)}")
print(f"\nnull counts per column:")
print(df.isnull().sum().to_string())

rows    : 10831
columns : 20

null counts per column:
id                         0
imdb_id                   10
popularity                 0
budget                  5665
revenue                 5983
original_title             0
cast                      76
director                  42
tagline                 2797
keywords                1476
overview                   2
runtime                    0
genres                    22
production_companies    1011
release_date            6310
vote_count                 0
vote_average               0
release_year            6310
budget_adj              5665
revenue_adj             5983


## Summary of Changes

| Step | Issue | Action |
|---|---|---|
| 1 | 2 duplicate rows | Dropped |
| 2 | Rows with all critical fields null | Dropped |
| 3 | `id` stored as float | Cast to `Int64` |
| 4 | Mixed `release_date` formats | Unified to `datetime64`; rebuilt `release_year` |
| 5 | Zeros in budget/revenue = "unknown" | Replaced with `NaN` |
| 6 | `vote_count`, `runtime` stored as float | Cast to `Int64` |
| 7 | Inconsistent spacing in pipe-delimited columns | Stripped per token |
| 8 | Whitespace in string columns | Stripped with `.str.strip()` |
| 9 | Empty strings after stripping | Replaced with `NaN` |
| 10 | `homepage` — 73% null | Column dropped |
| 11 | Impossible `runtime` values | Rows outside 1–600 min removed |
| 12 | `vote_average` outside 0–10 | Rows removed |
| 13 | Gaps in index after row drops | Index reset |

## 16. Save cleaned dataset

In [32]:
# save cleaned dataset
df.to_csv("Hey-pythonista_cleaned.csv", index=False)

print("\ncleaned dataset saved to imdb_cleaned.csv")


cleaned dataset saved to imdb_cleaned.csv
